# 01 — Quickstart & our data

Explore the events and profiles produced by the intake step (`out/data/`).
This notebook used to generate a synthetic banking book; it now reads our own
data instead.

> pragmatiq is an independent implementation inspired by the PRAGMA paper
> (arXiv 2604.08649) and is not affiliated with or endorsed by Revolut.

## Setup

Keep imports and notebook-level constants in one place so later cells focus on
the data workflow rather than environment setup.

In [ ]:
import json
from pathlib import Path

import pyarrow.parquet as pq
from IPython.display import display

from pragmatiq import api

# Locate the intake output (out/data) whether run from the repo root or notebooks/
ROOT = Path.cwd()
if not (ROOT / "out" / "data").exists():
    ROOT = ROOT.parent
RAW = ROOT / "out" / "data"   # events.parquet + profiles.parquet from the intake step

WORK = Path("runs_quickstart")

## Intake summary

The data is already built by the intake step, so there is no generator to run.
This cell shows what intake wrote.

In [ ]:
report = json.loads((RAW / "output_report.json").read_text())

display({k: report[k] for k in ("rows_in", "events_out", "customers", "profiles_out", "by_source")})

## Events and labels

Events are key–value–time records across four sources
(transaction / app / trading / communication). Our intake output has
`events.parquet` and `profiles.parquet`. Labels would live in `labels/*.parquet`;
we do not have any yet, so the label cell below is skipped until they exist.

In [ ]:
events = pq.ParquetFile(RAW / "events.parquet").read_row_group(0).slice(0, 8).to_pandas()

display(events[["user_id", "ts", "source", "fields"]])

In [ ]:
label_path = RAW / "labels" / "default_12m.parquet"
if label_path.exists():
    labels = pq.read_table(label_path).to_pandas()
    print("default_12m rows:", len(labels), "positives:", int(labels.label.sum()))
else:
    print("No labels yet:", label_path, "- build one (user_id, eval_ts, label int8) to use notebook 03.")

## Hand-crafted GBDT baseline

The project baseline (`scripts/baselines/credit_gbdt.py`) reaches
ROC-AUC ~0.78–0.83 on credit, which is the bar a pragmatiq probe should
approach.